In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# =====================================================
# 1. LOAD DATASET
# =====================================================

file_path = "dataset.csv"   # change dataset name

df = pd.read_csv(file_path)

print("Dataset Loaded")
print(df.head())


In [ ]:
# =====================================================
# 2. HANDLE MISSING VALUES
# =====================================================

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].mean())

for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# =====================================================
# 3. ENCODE CATEGORICAL COLUMNS
# =====================================================

encoder = LabelEncoder()

for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = encoder.fit_transform(df[col])


In [ ]:
# =====================================================
# 4. SELECT TARGET COLUMN
# =====================================================

target_column = df.columns[-1]

X = df.drop(columns=[target_column])
y = df[target_column]

In [ ]:
# =====================================================
# 5. NORMALIZATION
# =====================================================

scaler = StandardScaler()

X = scaler.fit_transform(X)

In [ ]:
# =====================================================
# 6. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# =====================================================
# 7. CREATE CLIENT SPLITS
# =====================================================

num_clients = 3

client_data = np.array_split(X_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

print(f"\nDataset split among {num_clients} clients")

In [ ]:
# =====================================================
# 8. LOCAL CLIENT TRAINING
# =====================================================

client_weights = []

for i in range(num_clients):

    print(f"\nTraining Client {i+1}")

    model = LogisticRegression(max_iter=1000)

    model.fit(client_data[i], client_labels[i])

    # Get local weights
    weights = model.coef_

In [ ]:
# =================================================
# 9. DIFFERENTIAL PRIVACY
# =================================================

noise = np.random.normal(0, 0.1, weights.shape)

noisy_weights = weights + noise

client_weights.append(noisy_weights)

print("Noise added to client weights")

In [ ]:
# =====================================================
# 10. FEDERATED AVERAGING
# =====================================================

global_weights = np.mean(client_weights, axis=0)

print("\nFederated Averaging Completed")

In [ ]:
# =====================================================
# 11. CREATE GLOBAL MODEL
# =====================================================

global_model = LogisticRegression(max_iter=1000)

# Initialize model
global_model.fit(X_train, y_train)

# Replace with federated weights
global_model.coef_ = global_weights

In [ ]:
# =====================================================
# 12. TEST GLOBAL MODEL
# =====================================================

predictions = global_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("\nGlobal Model Accuracy:", accuracy)